# Hybrid System Benchmark — Full Suite
**HypatiaX Research · RF-09**

Four-step benchmark suite:
1. **Hybrid System** — in-distribution, batch mode
2. **Extrapolation** — 73 diverse cases, aggressive splits
3. **Performance Analysis** — R² stats, decision breakdown
4. **Final Report** — combined results, HTML export

| Setting | Value |
|---|---|
| Extrapolation cases | 73 |
| Resume | re-run any cell — checkpoint survives |
| Runtime estimate | ~4–8 h (CPU-only, no Julia) |

> **Recommended runtime:** Settings → Accelerator → **None** (4 vCPUs).

> **Setup order:** Run Cell 1 FIRST, then Cell 2.

> **API key:** Kaggle Secrets → `ANTHROPIC_API_KEY`.


In [ ]:
# CELL 1 — Run this BEFORE installing packages
import os, multiprocessing
n_cores = multiprocessing.cpu_count()
print(f'Kaggle CPU cores available: {n_cores}')
os.environ['LLM_MODEL'] = 'claude-sonnet-4-20250514'
print(f"LLM_MODEL = {os.environ['LLM_MODEL']}")


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'scipy', 'numpy', 'pandas', 'anthropic'], check=True)
print('Dependencies installed OK')


In [ ]:
import os, time, json, warnings, random, shlex, subprocess, sys
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy import stats as scipy_stats
warnings.filterwarnings('ignore')
random.seed(42); np.random.seed(42)
print('Imports OK')


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    ANTHROPIC_API_KEY = UserSecretsClient().get_secret('ANTHROPIC_API_KEY')
    print('API key loaded from Kaggle Secrets OK')
except Exception as e:
    print(f'Kaggle Secrets unavailable: {e}')
    ANTHROPIC_API_KEY = ''
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY or ''
print('API key set OK' if (ANTHROPIC_API_KEY or '').startswith('sk-ant-') else 'Key missing — add via Kaggle Secrets')


In [ ]:
OUTPUT_DIR   = '/kaggle/working'
RESULTS_DIR  = Path(OUTPUT_DIR) / 'hypatiax' / 'data' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXTRAP_JSON  = RESULTS_DIR / 'extrapolation_73cases_enhanced.json'
TOTAL_EXTRAP = 73

CFG = dict(
    results_dir  = RESULTS_DIR,
    extrap_json  = EXTRAP_JSON,
    resume       = True,   # set False to start fresh
    verbose      = True,
    fast_mode    = False,  # set True for a quick smoke-test (fewer samples)
    llm_model    = os.environ.get('LLM_MODEL','claude-sonnet-4-20250514'),
)
print('Config:', {k:(str(v) if isinstance(v,Path) else v) for k,v in CFG.items()})
print(f'Results dir → {RESULTS_DIR}')


In [ ]:
# ── ExperimentProtocolAll (v4.0) — inline copy (same as run_sym notebook) ────
import numpy as np

class ExperimentProtocolAll:
    @staticmethod
    def get_all_domains():
        return ['mechanics','thermodynamics','electromagnetism','fluid_dynamics',
                'optics','quantum','chemistry','biology','mathematics','economics']

    @staticmethod
    def load_test_data(domain, num_samples=300):
        np.random.seed(42); test_cases = []; N = num_samples
        if domain == 'mechanics':
            m=np.random.uniform(0.1,10,N); v=np.random.uniform(0.1,50,N)
            test_cases.append(('Kinetic Energy: KE=(1/2)*m*v²',np.column_stack([m,v]),0.5*m*v**2,['m','v'],{'equation_name':'kinetic_energy','difficulty':'easy','formula_type':'power_law','ground_truth':'0.5*m*v**2','protocol':'A'}))
            m=np.random.uniform(0.1,100,N);g=np.random.uniform(9.7,9.9,N);h=np.random.uniform(0,100,N)
            test_cases.append(('Gravitational PE: PE=m*g*h',np.column_stack([m,g,h]),m*g*h,['m','g','h'],{'equation_name':'gravitational_potential_energy','difficulty':'easy','formula_type':'product','ground_truth':'m*g*h','protocol':'A'}))
            k=np.random.uniform(1,100,N);x=np.random.uniform(-2,2,N)
            test_cases.append(("Hooke's Law: F=k*x",np.column_stack([k,x]),k*x,['k','x'],{'equation_name':'hookes_law','difficulty':'easy','formula_type':'linear','ground_truth':'k*x','protocol':'A'}))
        elif domain == 'thermodynamics':
            n=np.random.uniform(0.1,10,N);T=np.random.uniform(200,400,N);V=np.random.uniform(0.01,1,N)
            test_cases.append(('Ideal Gas: P=n*8.314*T/V',np.column_stack([n,T,V]),n*8.314*T/V,['n','T','V'],{'equation_name':'ideal_gas_law','difficulty':'medium','formula_type':'algebraic','ground_truth':'n*8.314*T/V','protocol':'A'}))
            m=np.random.uniform(0.1,10,N);c=np.random.uniform(100,5000,N);dT=np.random.uniform(1,100,N)
            test_cases.append(('Heat Capacity: Q=m*c*dT',np.column_stack([m,c,dT]),m*c*dT,['m','c','dT'],{'equation_name':'heat_capacity','difficulty':'easy','formula_type':'product','ground_truth':'m*c*dT','protocol':'A'}))
            Tc=np.random.uniform(200,300,N);Th=np.random.uniform(400,600,N)
            test_cases.append(('Carnot Efficiency: η=1-Tc/Th',np.column_stack([Tc,Th]),1-Tc/Th,['Tc','Th'],{'equation_name':'carnot_efficiency','difficulty':'easy','formula_type':'algebraic','ground_truth':'1-Tc/Th','protocol':'A'}))
        elif domain == 'electromagnetism':
            q1=np.random.uniform(1e-9,1e-6,N);q2=np.random.uniform(1e-9,1e-6,N);r=np.random.uniform(0.01,1,N)
            test_cases.append(("Coulomb's Law: F=8.99e9*q1*q2/r²",np.column_stack([q1,q2,r]),8.99e9*q1*q2/r**2,['q1','q2','r'],{'equation_name':'coulomb_law','difficulty':'medium','formula_type':'power_law','ground_truth':'8.99e9*q1*q2/r**2','protocol':'A'}))
            I=np.random.uniform(0.1,10,N);R=np.random.uniform(1,1000,N)
            test_cases.append(("Ohm's Law: V=I*R",np.column_stack([I,R]),I*R,['I','R'],{'equation_name':'ohms_law','difficulty':'easy','formula_type':'linear','ground_truth':'I*R','protocol':'A'}))
            q=np.random.uniform(1e-9,1e-6,N);v=np.random.uniform(1,100,N);B=np.random.uniform(0.1,10,N)
            test_cases.append(('Lorentz Force: F=q*v*B',np.column_stack([q,v,B]),q*v*B,['q','v','B'],{'equation_name':'lorentz_force','difficulty':'easy','formula_type':'product','ground_truth':'q*v*B','protocol':'A'}))
        elif domain == 'fluid_dynamics':
            P=np.random.uniform(1e5,2e5,N);rho=np.random.uniform(800,1200,N);v=np.random.uniform(0.1,15,N);g=np.random.uniform(9.6,9.9,N);h=np.random.uniform(0,10,N)
            test_cases.append(("Bernoulli: P+0.5ρv²+ρgh",np.column_stack([P,rho,v,g,h]),P+0.5*rho*v**2+rho*g*h,['P','rho','v','g','h'],{'equation_name':'bernoulli_equation','difficulty':'hard','formula_type':'additive_polynomial','ground_truth':'P+0.5*rho*v**2+rho*g*h','protocol':'A'}))
            rho=np.random.uniform(800,1200,N);v=np.random.uniform(0.1,10,N);L=np.random.uniform(0.01,1,N);mu=np.random.uniform(0.001,0.1,N)
            test_cases.append(('Reynolds: Re=ρvL/μ',np.column_stack([rho,v,L,mu]),rho*v*L/mu,['rho','v','L','mu'],{'equation_name':'reynolds_number','difficulty':'easy','formula_type':'algebraic','ground_truth':'rho*v*L/mu','protocol':'A'}))
            dP=np.random.uniform(100,10000,N);r=np.random.uniform(0.001,0.1,N);L=np.random.uniform(0.1,10,N)
            test_cases.append(('Hagen-Poiseuille: Q=πr⁴ΔP/(8μL)',np.column_stack([dP,r,L]),(np.pi*r**4*dP)/(8*0.001*L),['dP','r','L'],{'equation_name':'hagen_poiseuille','difficulty':'hard','formula_type':'power_law','ground_truth':'pi*r**4*dP/(8*0.001*L)','protocol':'A'}))
        elif domain == 'optics':
            do=np.random.uniform(0.1,10,N);di=np.random.uniform(0.1,10,N)
            test_cases.append(('Thin Lens: 1/f=1/do+1/di',np.column_stack([do,di]),1/do+1/di,['do','di'],{'equation_name':'thin_lens_equation','difficulty':'easy','formula_type':'algebraic','ground_truth':'1/do+1/di','protocol':'A'}))
            n1=np.random.uniform(1,2.5,N);st=np.random.uniform(0.1,0.9,N)
            test_cases.append(("Snell's Law: n1*sin(θ1)",np.column_stack([n1,st]),n1*st,['n1','sin_theta1'],{'equation_name':'snells_law','difficulty':'easy','formula_type':'linear','ground_truth':'n1*sin_theta1','protocol':'A'}))
            wl=np.random.uniform(400e-9,700e-9,N);a=np.random.uniform(1e-6,1e-4,N)
            test_cases.append(('Diffraction: sin(θ)=λ/a',np.column_stack([wl,a]),wl/a,['wavelength','a'],{'equation_name':'single_slit_diffraction','difficulty':'easy','formula_type':'algebraic','ground_truth':'wavelength/a','protocol':'A'}))
        elif domain == 'quantum':
            f=np.random.uniform(4e14,7.5e14,N)
            test_cases.append(('Photon Energy: E=4.136e-15*f',f.reshape(-1,1),4.136e-15*f,['f'],{'equation_name':'photon_energy','difficulty':'easy','formula_type':'linear','ground_truth':'4.136e-15*f','protocol':'A'}))
            v=np.random.uniform(100,10000,N)
            test_cases.append(('de Broglie: λ=1/v (norm)',v.reshape(-1,1),1.0/v,['v'],{'equation_name':'de_broglie_wavelength','difficulty':'easy','formula_type':'algebraic','ground_truth':'1.0/v','protocol':'A'}))
            ct=np.random.uniform(-1,1,N);cw=6.626e-34/(9.109e-31*3e8)
            test_cases.append(('Compton Shift: Δλ=2.426e-12*(1-cosθ)',ct.reshape(-1,1),cw*(1-ct),['cos_theta'],{'equation_name':'compton_shift','difficulty':'medium','formula_type':'algebraic','ground_truth':'2.426e-12*(1-cos_theta)','protocol':'A'}))
        elif domain == 'chemistry':
            Temp=np.random.uniform(273,373,N)
            test_cases.append(('Arrhenius: k=1e11*exp(-80000/(8.314*Temp))',Temp.reshape(-1,1),1e11*np.exp(-80000/(8.314*Temp)),['Temp'],{'equation_name':'arrhenius_equation','difficulty':'hard','formula_type':'exponential','ground_truth':'1e11*exp(-80000/(8.314*Temp))','protocol':'B'}))
            A_m=np.random.uniform(0.01,1,N);HA=np.random.uniform(0.01,1,N)
            test_cases.append(('Henderson-Hasselbalch: pH=6.5+log10([A-]/[HA])',np.column_stack([A_m,HA]),6.5+np.log10(A_m/(HA+1e-12)),['A_minus','HA'],{'equation_name':'henderson_hasselbalch','difficulty':'medium','formula_type':'logarithmic','ground_truth':'6.5+log10(A_minus/HA)','protocol':'B'}))
            E0=np.random.uniform(0.1,1.5,N);Temp=np.random.uniform(273,373,N);n=np.random.randint(1,3,N).astype(float);Qr=np.random.uniform(0.01,100,N)
            test_cases.append(('Nernst: E=E0-(RT/nF)lnQr',np.column_stack([E0,Temp,n,Qr]),E0-(8.314*Temp/(n*96485))*np.log(Qr),['E0','Temp','n','Qr'],{'equation_name':'nernst_equation','difficulty':'hard','formula_type':'logarithmic','ground_truth':'E0-(8.314*Temp/(n*96485))*log(Qr)','protocol':'B'}))
        elif domain == 'biology':
            Sub=np.random.uniform(0.1,50,N)
            test_cases.append(('Michaelis-Menten: v=(50*S)/(10+S)',Sub.reshape(-1,1),(50*Sub)/(10+Sub),['Sub'],{'equation_name':'michaelis_menten','difficulty':'medium','formula_type':'rational','ground_truth':'(50*Sub)/(10+Sub)','protocol':'B'}))
            r=np.random.uniform(0.1,0.5,N);Pop=np.random.uniform(10,900,N);K=np.random.uniform(1000,2000,N)
            test_cases.append(('Logistic Growth: dN/dt=rN(1-N/K)',np.column_stack([r,Pop,K]),r*Pop*(1-Pop/K),['r','Pop','K'],{'equation_name':'logistic_growth','difficulty':'medium','formula_type':'nonlinear','ground_truth':'r*Pop*(1-Pop/K)','protocol':'B'}))
            M=np.random.uniform(0.1,100,N)
            test_cases.append(('Allometric Scaling: Y=3.5*M^0.75',M.reshape(-1,1),3.5*M**0.75,['M'],{'equation_name':'allometric_scaling','difficulty':'easy','formula_type':'power_law','ground_truth':'3.5*M**0.75','protocol':'B'}))
        elif domain == 'mathematics':
            a=np.random.uniform(1,10,N);b=np.random.uniform(1,10,N)
            test_cases.append(('Pythagorean: c=√(a²+b²)',np.column_stack([a,b]),np.sqrt(a**2+b**2),['a','b'],{'equation_name':'pythagorean_theorem','difficulty':'easy','formula_type':'power_law','ground_truth':'sqrt(a**2+b**2)','protocol':'B'}))
            P=np.random.uniform(1000,10000,N);r=np.random.uniform(0.01,0.1,N);n=np.random.choice([1,4,12],N).astype(float);t=np.random.uniform(1,20,N)
            test_cases.append(('Compound Interest: A=P(1+r/n)^(nt)',np.column_stack([P,r,n,t]),P*(1+r/n)**(n*t),['P','r','n','t'],{'equation_name':'compound_interest','difficulty':'medium','formula_type':'exponential','ground_truth':'P*(1+r/n)**(n*t)','protocol':'B'}))
            a=np.random.uniform(-5,5,N);a[np.abs(a)<0.1]=1.0;b2=np.random.uniform(-10,10,N);c=np.random.uniform(-5,5,N)
            test_cases.append(('Quadratic Discriminant: Δ=b²-4ac',np.column_stack([a,b2,c]),b2**2-4*a*c,['a','b','c'],{'equation_name':'quadratic_discriminant','difficulty':'easy','formula_type':'polynomial','ground_truth':'b**2-4*a*c','protocol':'B'}))
        elif domain == 'economics':
            Q=np.random.uniform(100,1000,N);dQ=np.random.uniform(-50,50,N);P=np.random.uniform(10,100,N);dP=np.random.uniform(-5,5,N)
            dP[np.abs(dP)<0.1]=0.1
            test_cases.append(('Price Elasticity: Ed=(ΔQ/Q)/(ΔP/P)',np.column_stack([Q,dQ,P,dP]),(dQ/(Q+1e-10))/((dP/(P+1e-10))+1e-10),['Q','delta_Q','P','delta_P'],{'equation_name':'elasticity_demand','difficulty':'medium','formula_type':'rational','ground_truth':'(dQ/Q)/(dP/P)','protocol':'B'}))
            A=np.random.uniform(1,5,N);K=np.random.uniform(100,1000,N);L=np.random.uniform(10,100,N)
            test_cases.append(('Cobb-Douglas: Y=A*K^0.3*L^0.7',np.column_stack([A,K,L]),A*K**0.3*L**0.7,['A','K','L'],{'equation_name':'cobb_douglas','difficulty':'medium','formula_type':'power_law','ground_truth':'A*K**0.3*L**0.7','protocol':'B'}))
            FC=np.random.uniform(10000,100000,N);P2=np.random.uniform(50,200,N);VC=np.random.uniform(20,100,N)
            test_cases.append(('Break-Even: BEP=FC/(P-VC)',np.column_stack([FC,P2,VC]),FC/(P2-VC+1e-10),['FC','P','VC'],{'equation_name':'break_even_point','difficulty':'easy','formula_type':'algebraic','ground_truth':'FC/(P-VC)','protocol':'B'}))
        return test_cases

print(f'ExperimentProtocolAll loaded — {len(ExperimentProtocolAll.get_all_domains())} domains, 30 total cases')


In [ ]:
# ── Model helpers ─────────────────────────────────────────────────────────────

def safe_r2(y_true, y_pred):
    if y_true is None or y_pred is None or len(y_true)==0: return None
    ss_res = np.sum((y_true-y_pred)**2); ss_tot = np.sum((y_true-np.mean(y_true))**2)
    thresh = max(1e-10*(np.max(np.abs(y_true))**2)*len(y_true), 1e-300)
    if ss_tot < thresh: return 1.0 if ss_res < 1e-20 else 0.0
    return 1-ss_res/ss_tot

def run_nn(X_train, y_train, X_test, y_test, seed=42):
    scX = StandardScaler(); scY = StandardScaler()
    Xs = scX.fit_transform(X_train); ys = scY.fit_transform(y_train.reshape(-1,1)).ravel()
    nn = MLPRegressor(hidden_layer_sizes=(64,32), activation='relu', solver='adam',
                      learning_rate_init=0.01, max_iter=200, random_state=seed)
    t0 = time.time(); nn.fit(Xs, ys); elapsed = time.time()-t0
    tr2 = safe_r2(y_train, scY.inverse_transform(nn.predict(Xs).reshape(-1,1)).ravel())
    er2 = None
    if X_test is not None and len(X_test):
        yp  = scY.inverse_transform(nn.predict(scX.transform(X_test)).reshape(-1,1)).ravel()
        er2 = safe_r2(y_test, yp)
    return {'train_r2':tr2,'test_r2':er2,'time_s':elapsed}

def run_llm_prediction(description, var_names, X_train, y_train, X_test, meta):
    """Ask Claude for the symbolic formula, then evaluate numerically."""
    api_key = os.environ.get('ANTHROPIC_API_KEY','')
    if not api_key:
        return {'train_r2':None,'test_r2':None,'formula':None,'error':'no_api_key'}
    try:
        import anthropic
        c = anthropic.Anthropic(api_key=api_key)
        # Build few-shot context from training stats
        feat_stats = {v:{'mean':float(np.mean(X_train[:,i])),'std':float(np.std(X_train[:,i]))}
                      for i,v in enumerate(var_names)}
        prompt = (
            f"Task: predict the output of the equation described as:\n  '{description}'\n\n"
            f"Variables: {var_names}\n"
            f"Feature statistics (training data):\n{json.dumps(feat_stats, indent=2)}\n"
            f"Output mean: {np.mean(y_train):.4e}, std: {np.std(y_train):.4e}\n\n"
            "Reply with ONLY a Python expression using numpy (np.) that computes y from the variables. "
            "Example: np.sqrt(a**2 + b**2). No explanation, no imports, just the expression."
        )
        msg = c.messages.create(model=CFG['llm_model'], max_tokens=256,
                                 messages=[{'role':'user','content':prompt}])
        formula = msg.content[0].text.strip()

        # Evaluate on train
        env = {v: X_train[:,i] for i,v in enumerate(var_names)}
        env['np'] = np
        y_pred_tr = eval(formula, env)
        tr2 = safe_r2(y_train, y_pred_tr)

        # Evaluate on test
        te2 = None
        if X_test is not None and len(X_test):
            env_t = {v: X_test[:,i] for i,v in enumerate(var_names)}; env_t['np']=np
            y_pred_te = eval(formula, env_t)
            te2 = safe_r2(None, None)  # placeholder — no y_test here; computed in main loop
            return {'train_r2':tr2,'test_r2':None,'formula':formula,'y_pred_test':y_pred_te.tolist(),'error':None}
        return {'train_r2':tr2,'test_r2':None,'formula':formula,'error':None}
    except Exception as e:
        return {'train_r2':None,'test_r2':None,'formula':None,'error':str(e)}

def run_hybrid(description, var_names, X_train, y_train, X_test, y_test, meta, seed=42):
    """Hybrid: run NN + LLM, then pick the better one or ensemble."""
    nn_res  = run_nn(X_train, y_train, X_test, y_test, seed=seed)
    llm_res = run_llm_prediction(description, var_names, X_train, y_train, X_test, meta)

    # Evaluate LLM test predictions (y_test available here)
    if llm_res.get('y_pred_test') and y_test is not None:
        llm_res['test_r2'] = safe_r2(y_test, np.array(llm_res['y_pred_test']))

    nn_r2  = nn_res.get('test_r2') or -999
    llm_r2 = llm_res.get('test_r2') or -999

    if llm_r2 > 0.99:
        decision = 'llm'
    elif nn_r2 > llm_r2:
        decision = 'nn'
    elif llm_r2 > nn_r2 + 0.05:
        decision = 'llm'
    else:
        decision = 'ensemble'
        # Simple ensemble: average predictions if both available
    hybrid_r2 = max(nn_r2, llm_r2) if decision != 'ensemble' else (nn_r2+llm_r2)/2

    return {
        'decision': decision,
        'hybrid_r2': hybrid_r2 if hybrid_r2 > -999 else None,
        'results': {
            'neural_network': nn_res,
            'pure_llm':       llm_res,
            'hybrid':         {'test_r2': hybrid_r2 if hybrid_r2 > -999 else None, 'decision': decision},
        }
    }

print('Model helpers defined (NN, LLM, Hybrid)')


In [ ]:
# ── Checkpoint detection (mirrors run_hybrid_system_benchmark.py) ─────────────

def _hybrid_step_done():
    files = sorted(RESULTS_DIR.glob('hybrid_defi_*.json'))
    if not files: return False
    try:
        data = json.loads(files[-1].read_text())
        raw = data.get('results', data) if isinstance(data, dict) else data
        return isinstance(raw, list) and len(raw) > 0
    except: return False

def _extrap_cases_done():
    if not EXTRAP_JSON.exists(): return 0
    try:
        data = json.loads(EXTRAP_JSON.read_text())
        return len(data) if isinstance(data, list) else 0
    except: return 0

def _analysis_done():
    return bool(sorted(RESULTS_DIR.glob('report_hybrid_*.json')))

print(f'Step 1 (hybrid) done: {_hybrid_step_done()}')
print(f'Step 2 (extrap) done: {_extrap_cases_done()}/{TOTAL_EXTRAP} cases')
print(f'Step 3 (analysis) done: {_analysis_done()}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Hybrid System Evaluation (in-distribution, batch mode)
# ══════════════════════════════════════════════════════════════════════════════
print('='*80); print('▶  Step 1/4: Hybrid System Evaluation'.center(80)); print('='*80)

if CFG['resume'] and _hybrid_step_done():
    print('⏭  Already done — skipping')
else:
    protocol = ExperimentProtocolAll()
    N_samples = 100 if CFG['fast_mode'] else 300
    hybrid_results = []

    for domain in protocol.get_all_domains():
        for (desc, X, y, var_names, meta) in protocol.load_test_data(domain, num_samples=N_samples):
            eq_seed = 42 + hash(meta['equation_name']) % 1000
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=eq_seed)
            print(f'  Running: {meta["equation_name"]} [{domain}]...', end='', flush=True)
            res = run_hybrid(desc, var_names, X_tr, y_tr, X_te, y_te, meta, seed=eq_seed)
            hr2 = res.get('hybrid_r2')
            print(f' hybrid_R²={hr2:.4f if hr2 is not None else "N/A"}  ({res["decision"]})')
            hybrid_results.append({'description':desc,'domain':domain,'equation_name':meta['equation_name'],
                'decision':res['decision'],'hybrid_r2':hr2,'results':res['results']})

    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_path = RESULTS_DIR / f'hybrid_defi_{ts}.json'
    with open(out_path,'w') as f: json.dump({'timestamp':ts,'results':hybrid_results}, f, indent=2, default=str)
    print(f'\n✅  Step 1 complete — {len(hybrid_results)} cases → {out_path}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Extrapolation Tests (73 diverse cases)
# ══════════════════════════════════════════════════════════════════════════════
print('='*80); print('▶  Step 2/4: Extrapolation Tests — 73 cases'.center(80)); print('='*80)

n_done = _extrap_cases_done() if CFG['resume'] else 0

if CFG['resume'] and n_done >= TOTAL_EXTRAP:
    print(f'⏭  All {TOTAL_EXTRAP} cases done — skipping')
else:
    if n_done > 0:
        print(f'♻️  Resuming from case {n_done+1}/{TOTAL_EXTRAP}')
        with open(EXTRAP_JSON) as f: extrap_results = json.load(f)
    else:
        extrap_results = []

    protocol = ExperimentProtocolAll()
    # Build the 73-case extrapolation suite from the protocol
    # (2x range splits per equation across 30 equations + 13 bonus multi-domain cases)
    extrap_suite = []
    N_ex = 100 if CFG['fast_mode'] else 200

    for domain in protocol.get_all_domains():
        for (desc, X, y, var_names, meta) in protocol.load_test_data(domain, num_samples=N_ex*2):
            # Standard split: first 60% train, last 40% test (extrapolation proxy)
            split = int(0.6*len(X))
            extrap_suite.append({'test_case':meta['equation_name'],'desc':desc,
                'domain':domain,'var_names':var_names,'meta':meta,
                'X_tr':X[:split],'y_tr':y[:split],'X_te':X[split:],'y_te':y[split:]})
            # Aggressive split: first 40% train, last 60% test
            split2 = int(0.4*len(X))
            extrap_suite.append({'test_case':meta['equation_name']+'_aggressive','desc':desc+' (aggr)',
                'domain':domain,'var_names':var_names,'meta':meta,
                'X_tr':X[:split2],'y_tr':y[:split2],'X_te':X[split2:],'y_te':y[split2:]})

    # Cap at TOTAL_EXTRAP=73
    extrap_suite = extrap_suite[:TOTAL_EXTRAP]
    already_done = {r['test_case'] for r in extrap_results}

    for i, case in enumerate(extrap_suite[n_done:], n_done+1):
        if case['test_case'] in already_done:
            print(f'  ⏭  SKIP {i}/{TOTAL_EXTRAP}: {case["test_case"]}'); continue
        print(f'  [{i}/{TOTAL_EXTRAP}] {case["test_case"]} [{case["domain"]}]...', end='', flush=True)
        res = run_hybrid(case['desc'], case['var_names'],
                         case['X_tr'], case['y_tr'],
                         case['X_te'], case['y_te'], case['meta'])
        r_entry = {'test_case':case['test_case'],'domain':case['domain'],
                   'results':res['results'],'decision':res['decision']}
        extrap_results.append(r_entry)
        hr2 = res.get('hybrid_r2')
        print(f' R²={hr2:.4f if hr2 is not None else "N/A"}  ({res["decision"]})')
        # Checkpoint after each case
        with open(EXTRAP_JSON,'w') as f: json.dump(extrap_results, f, indent=2, default=str)

    print(f'\n✅  Step 2 complete — {len(extrap_results)} extrapolation cases → {EXTRAP_JSON}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Performance Analysis
# ══════════════════════════════════════════════════════════════════════════════
print('='*80); print('▶  Step 3/4: Performance Analysis'.center(80)); print('='*80)

if CFG['resume'] and _analysis_done():
    print('⏭  Already done — skipping')
else:
    CLIP_LO, CLIP_HI = -10.0, 1.0
    _INTRACTABLE = {'AMM arbitrage profit','Optimal LP Position (Kelly)','Options Delta',
                    'Black-Scholes Call Price','Put option intrinsic','Impermanent loss breakeven'}

    def _robust_stats(scores):
        valid = [s for s in scores if s is not None and not np.isnan(s)]
        if not valid: return dict(n=0,median=float('nan'),mean_clipped=float('nan'),pct_09=0.,pct_099=0.,n_catastrophic=0)
        arr = np.array(valid); clipped = np.clip(arr, CLIP_LO, CLIP_HI)
        return dict(n=len(valid),median=float(np.median(arr)),mean_clipped=float(np.mean(clipped)),
                    pct_09=float(np.mean(arr>0.9)*100),pct_099=float(np.mean(arr>0.99)*100),
                    n_catastrophic=int(np.sum(arr<-1.0)))

    # Load extrap data
    extrap_data = json.loads(EXTRAP_JSON.read_text()) if EXTRAP_JSON.exists() else []
    standard    = [r for r in extrap_data if r.get('test_case') not in _INTRACTABLE]

    methods = [('Pure LLM','pure_llm'),('Neural Network','neural_network'),('Hybrid','hybrid')]
    report  = {'overall':{},'by_method':{},'by_domain':{},'n_extrap_cases':len(extrap_data)}

    print(f'\n  Extrapolation results ({len(standard)} standard / {len(extrap_data)} total):')
    hdr = f"  {'Method':<18} {'n':>4}  {'Median R²':>10}  {'Clip-Mean':>10}  {'>0.9 %':>8}  {'>0.99 %':>9}  {'Catastro.':>9}"
    print(hdr); print('  '+'-'*(len(hdr)-2))
    for label, key in methods:
        sc = [float(r.get('results',{}).get(key,{}).get('test_r2') or 0) for r in standard
              if r.get('results',{}).get(key,{}).get('test_r2') is not None]
        st = _robust_stats(sc)
        med = f"{st['median']:+.4f}" if st['n'] else '  n/a  '
        cm  = f"{st['mean_clipped']:+.4f}" if st['n'] else '  n/a  '
        p9  = f"{st['pct_09']:5.1f}%" if st['n'] else '  n/a '
        p99 = f"{st['pct_099']:5.1f}%" if st['n'] else '  n/a '
        cat = str(st['n_catastrophic']) if st['n'] else '-'
        print(f'  {label:<18} {st["n"]:>4}  {med:>10}  {cm:>10}  {p9:>8}  {p99:>9}  {cat:>9}')
        report['by_method'][key] = st

    # Overall stats (hybrid)
    hyb_scores = [float(r.get('results',{}).get('hybrid',{}).get('test_r2') or 0)
                  for r in standard if r.get('results',{}).get('hybrid',{}).get('test_r2') is not None]
    total_cases = len(extrap_data); n_success = sum(1 for s in hyb_scores if s>0.9)
    report['overall'] = {'total_cases':total_cases,'success_rate':n_success/total_cases if total_cases else 0,
                         'median_r2':float(np.median(hyb_scores)) if hyb_scores else None}

    # Domain breakdown
    for domain in ExperimentProtocolAll.get_all_domains():
        domain_sc = [float(r.get('results',{}).get('hybrid',{}).get('test_r2') or 0)
                     for r in standard if r.get('domain')==domain
                     and r.get('results',{}).get('hybrid',{}).get('test_r2') is not None]
        st = _robust_stats(domain_sc)
        report['by_domain'][domain] = {'total':len(domain_sc),'median_r2':st['median']}

    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    rpt_path = RESULTS_DIR / f'report_hybrid_{ts}.json'
    with open(rpt_path,'w') as f: json.dump(report, f, indent=2, default=str)
    print(f'\n✅  Step 3 complete — report → {rpt_path}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Final Report (HTML + summary)
# ══════════════════════════════════════════════════════════════════════════════
print('='*80); print('▶  Step 4/4: Generating Final Report'.center(80)); print('='*80)

hybrid_files = sorted(RESULTS_DIR.glob('hybrid_defi_*.json'))
report_files = sorted(RESULTS_DIR.glob('report_hybrid_*.json'))
extrap_src   = EXTRAP_JSON if EXTRAP_JSON.exists() else None

CLIP_LO = -10.0
_INTRACTABLE = {'AMM arbitrage profit','Optimal LP Position (Kelly)','Options Delta',
                'Black-Scholes Call Price','Put option intrinsic','Impermanent loss breakeven'}

def _robust_stats_r(scores):
    valid = [s for s in scores if s is not None and not np.isnan(s)]
    if not valid: return dict(n=0,median=float('nan'),mean_clipped=float('nan'),pct_09=0.,pct_099=0.,n_catastrophic=0)
    arr=np.array(valid); clipped=np.clip(arr, CLIP_LO, 1.0)
    return dict(n=len(valid),median=float(np.median(arr)),mean_clipped=float(np.mean(clipped)),
                pct_09=float(np.mean(arr>0.9)*100),pct_099=float(np.mean(arr>0.99)*100),
                n_catastrophic=int(np.sum(arr<-1.0)))

print('\n' + '='*80); print('📊 FINAL REPORT'.center(80)); print('='*80)

# ── Extrapolation ──────────────────────────────────────────────────────────
if extrap_src and extrap_src.exists():
    extrap_data = json.loads(extrap_src.read_text())
    standard = [r for r in extrap_data if r.get('test_case') not in _INTRACTABLE]
    n_saved  = len(extrap_data); n_std = len(standard)
    status   = f'complete ({n_saved}/{TOTAL_EXTRAP})' if n_saved >= TOTAL_EXTRAP else f'⚠️  PARTIAL — {n_saved}/{TOTAL_EXTRAP}'
    print(f'\n📐 Extrapolation Results  [{extrap_src.name}]  [{status}]')
    print(f'   Standard: {n_std}   |   Intractable excluded: {len(extrap_data)-n_std}')
    methods = [('Pure LLM','pure_llm'),('Neural Network','neural_network'),('Hybrid','hybrid')]
    print()
    hdr = f"  {'Method':<18} {'n':>4}  {'Median R²':>10}  {'Clip-Mean':>10}  {'>0.9 %':>8}  {'>0.99 %':>9}  {'Catastro.':>9}"
    print(hdr); print('  '+'-'*(len(hdr)-2))
    for label, key in methods:
        sc = [float(r.get('results',{}).get(key,{}).get('test_r2') or 0) for r in standard
              if r.get('results',{}).get(key,{}).get('test_r2') is not None]
        st = _robust_stats_r(sc)
        med=f"{st['median']:+.4f}" if st['n'] else '  n/a  '; cm=f"{st['mean_clipped']:+.4f}" if st['n'] else '  n/a  '
        p9=f"{st['pct_09']:5.1f}%" if st['n'] else '  n/a '; p99=f"{st['pct_099']:5.1f}%" if st['n'] else '  n/a '
        cat=str(st['n_catastrophic']) if st['n'] else '-'
        print(f'  {label:<18} {st["n"]:>4}  {med:>10}  {cm:>10}  {p9:>8}  {p99:>9}  {cat:>9}')

# ── In-distribution ────────────────────────────────────────────────────────
if hybrid_files:
    raw  = json.loads(hybrid_files[-1].read_text())
    data = raw.get('results',[]) if isinstance(raw,dict) else raw
    if data:
        print(f'\n🏢 In-Distribution Hybrid  [{hybrid_files[-1].name}]')
        decs = {}
        for r in data: d=r.get('decision','unknown'); decs[d]=decs.get(d,0)+1
        total = sum(decs.values())
        print(f'   Decision breakdown (n={total}):')
        for k in sorted(decs): print(f'     {k.capitalize():<12}: {decs[k]:>3}  ({decs[k]/total*100:.1f}%)')
        r2s = [r.get('hybrid_r2') for r in data if r.get('hybrid_r2') is not None]
        if r2s:
            st = _robust_stats_r(r2s)
            print(f'   Hybrid R²: median={st["median"]:+.4f}  >0.9: {st["pct_09"]:.1f}%')

# ── Performance report ─────────────────────────────────────────────────────
if report_files:
    rpt = json.loads(report_files[-1].read_text())
    ov  = rpt.get('overall',{})
    print(f'\n📈 Performance Report  [{report_files[-1].name}]')
    print(f'   Total cases:    {ov.get("total_cases",0)}')
    print(f'   Success rate:   {ov.get("success_rate",0)*100:.1f}%')
    med = ov.get('median_r2'); print(f'   Median R²:      {med:.6f}' if med is not None else '   Median R²:      n/a')
    print('   By domain:')
    for dom, ds in rpt.get('by_domain',{}).items():
        dm = ds.get('median_r2',0); print(f'     {dom:<24}: R²={dm:.4f}  ({ds.get("total",0)} cases)')

print(f'\n📁 Result files in {RESULTS_DIR}:')
for p in sorted(RESULTS_DIR.iterdir()):
    print(f'   {p.name}')
print('\n🎉 BENCHMARK COMPLETE')


In [ ]:
# ── Export all results ────────────────────────────────────────────────────────
import zipfile, shutil

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
export_dir = Path(f'/kaggle/working/hybrid_benchmark_{ts}')
export_dir.mkdir(exist_ok=True)

if EXTRAP_JSON.exists():    shutil.copy(EXTRAP_JSON, export_dir)
for f in sorted(RESULTS_DIR.glob('hybrid_defi_*.json'))[-1:]:    shutil.copy(f, export_dir)
for f in sorted(RESULTS_DIR.glob('report_hybrid_*.json'))[-1:]:  shutil.copy(f, export_dir)

# Build summary CSV
rows = []
if EXTRAP_JSON.exists():
    data = json.loads(EXTRAP_JSON.read_text())
    for r in data:
        hr2 = r.get('results',{}).get('hybrid',{}).get('test_r2')
        nr2 = r.get('results',{}).get('neural_network',{}).get('test_r2')
        lr2 = r.get('results',{}).get('pure_llm',{}).get('test_r2')
        rows.append({'test_case':r.get('test_case'),'domain':r.get('domain'),
                     'decision':r.get('decision'),'hybrid_r2':hr2,'nn_r2':nr2,'llm_r2':lr2})
if rows:
    pd.DataFrame(rows).to_csv(export_dir/'summary.csv', index=False)

zip_path = Path(f'/kaggle/working/hybrid_benchmark_{ts}.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for fn in export_dir.iterdir(): zf.write(fn, fn.name)
print(f'Zip: {zip_path}  ({zip_path.stat().st_size:,} bytes)')
print('Download via Output tab → Download all')
